# Valence Band Offset of the GaAs/AlAs (110) Strained Superlattice

> **Yoyo Hinuma, Andreas Grüneis, Georg Kresse and Fumiyasu Oba**, "Band alignment of semiconductors from
> density-functional theory and many-body perturbation theory", Physical Review B 90, 155405 (2014).
> [DOI:10.1103/PhysRevB.90.155405](https://doi.org/10.1103/PhysRevB.90.155405)

Computes the valence band offset of the GaAs/AlAs (110) interface from the superlattice and the two strained bulk
cells created in the [structure notebook](interface_3d_3d_gallium_arsenide_aluminum_arsenide.ipynb), with the
platform's [Valence Band Offset](../workflows/valence_band_offset.ipynb) workflow, and compares it with Hinuma et al.
(2014), Table II. As in the paper, each bulk's valence band maximum is measured from its own average potential V̄,
and the superlattice gives the step in V̄ between the two materials:

ΔE_v = (E_VBM − V̄)_GaAs − (E_VBM − V̄)_AlAs + (V̄_GaAs − V̄_AlAs)_superlattice

The paper relaxed the out-of-plane lattice constant and the atomic positions; this notebook runs the cell as built.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](interface_3d_3d_gallium_arsenide_aluminum_arsenide.ipynb), which saves `GaAs/AlAs (110) superlattice 12+12`, `GaAs (110) bulk strained` and `AlAs (110) bulk strained` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4, or use the defaults.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the results; the last section prints the comparison with the manuscript.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the superlattice and the two bulks by name from the `uploads` folder or the platform, print their provenance.
1. Configure the DFT model and a k-grid per material.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Configure the Valence Band Offset workflow: the model, a k-grid per material and the band structures the valence band maxima are read from.
1. Run the job, or reuse one that already ran on the same materials with the same settings.
1. Retrieve the valence band maxima and the potential profiles, compute the offset, and plot the potential with the bulk potentials overlaid.
1. Compare with Hinuma et al. (2014).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2. Material names

In [ ]:
# Names saved by interface_3d_3d_gallium_arsenide_aluminum_arsenide.ipynb.
SUPERLATTICE_NAME = "GaAs/AlAs (110) superlattice 12+12"
SUBSTRATE_BULK_NAME = "GaAs (110) bulk strained"
FILM_BULK_NAME = "AlAs (110) bulk strained"

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

VBO_WORKFLOW_SEARCH_TERM = "valence_band_offset.json"
MY_WORKFLOW_NAME = "Valence Band Offset"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 16  # the OR queue allows at most 16 cores per node
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
MODEL_SUBTYPE = "gga"
FUNCTIONAL = "pbe"  # Hinuma et al. use PBE
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the PBE family the platform publishes for Ga, Al and As
VALENCE_ELECTRONS_PER_ATOM = {"Ga": 19, "Al": 3, "As": 5}  # GBRV pseudopotentials, Ga with its 3s, 3p and 3d shells
BLOCK_LABEL = {SUBSTRATE_BULK_NAME: "GaAs", FILM_BULK_NAME: "AlAs"}
BLOCK_CATION = {SUBSTRATE_BULK_NAME: "Ga", FILM_BULK_NAME: "Al"}  # one cation per (110) plane marks each block
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff
ADDITIONAL_PARAMETERS = {"electrons": {"mixing_mode": "local-TF", "mixing_beta": 0.2, "electron_maxstep": 200}}  # the long cell needs local-TF mixing to converge
KPOINT_DENSITY = 3.5  # points per Å⁻¹: 6x4x1 on the superlattice as in the paper, an equivalent grid on the bulks
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY}"

SCF_UNIT = "pw_scf"
BANDS_UNIT = "pw_bands"
AVERAGE_UNIT = "average ESP"

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder and print provenance

The superlattice holds 12 (110) planes of each material, with both materials strained to the mean lattice constant.

In [ ]:
import numpy as np
from collections import Counter
from mat3ra.notebooks_utils.core.entity.material.api import load_material

MATERIAL_NAMES = [SUPERLATTICE_NAME, SUBSTRATE_BULK_NAME, FILM_BULK_NAME]
materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in MATERIAL_NAMES}


def get_plane_spacings(material):
    cartesian_material = material.clone()
    cartesian_material.to_cartesian()
    heights = np.array(cartesian_material.basis.coordinates.values)[:, 2]
    plane_heights = np.unique(np.round(heights, 4) % round(material.lattice.c, 4))
    return np.diff(np.append(plane_heights, plane_heights[0] + material.lattice.c))


valence_electrons = {}
for name, material in materials_by_name.items():
    composition = Counter(material.basis.elements.values)
    electron_count = sum(VALENCE_ELECTRONS_PER_ATOM[element] * count for element, count in composition.items())
    lattice = material.lattice
    valence_electrons[name] = electron_count
    print(f"{name}: {dict(composition)}, {material.basis.number_of_atoms} atoms, "
          f"cell {lattice.a:.4f} x {lattice.b:.4f} x {lattice.c:.4f} Å, {electron_count} valence electrons")
    print(f"  plane spacings along c: {np.round(get_plane_spacings(material), 4).tolist()} Å")

## 4. Configure the DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. k-grid per material

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

kgrid = {name: kgrid_from_density(material, KPOINT_DENSITY) for name, material in materials_by_name.items()}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Configure the Valence Band Offset workflow

The workflow runs the same band structure and electrostatic potential calculation on the superlattice and on the
two bulks, each with its own k-grid, and stores the band structures. Section 8 computes the offset from them and
from the planar-averaged potential, as the paper does, so the workflow's own lineup step is left out.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.context.providers import PointsGridDataProvider
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, patch_workflow_qe_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

superlattice = materials_by_name[SUPERLATTICE_NAME]
job_materials = [superlattice, materials_by_name[SUBSTRATE_BULK_NAME], materials_by_name[FILM_BULK_NAME]]

workflow = Workflow.create(
    WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(VBO_WORKFLOW_SEARCH_TERM)
)
workflow.name = f"{MY_WORKFLOW_NAME} {' | '.join(MATERIAL_NAMES)} {MODEL_TAG}"
for subworkflow in list(workflow.subworkflows):
    if subworkflow.application.name != APPLICATION_NAME:
        workflow.remove_unit(workflow.get_unit_by_name(name=subworkflow.name).flowchartId)
        workflow.remove_subworkflow_by_id(subworkflow.id)
for unit_name in [SCF_UNIT, BANDS_UNIT]:
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)
for subworkflow in workflow.subworkflows:
    subworkflow.model = model
    material_index = int(next(unit.value for unit in subworkflow.units if unit.name.startswith("Set Material Index")))
    material = job_materials[material_index]

    bands_unit = subworkflow.get_unit_by_name(name=BANDS_UNIT)
    bands_unit.results.append({"name": "band_structure"})
    subworkflow.set_unit(bands_unit)

    scf_unit = subworkflow.get_unit_by_name(name=SCF_UNIT)
    scf_unit.add_context(PointsGridDataProvider(material=material, dimensions=kgrid[MATERIAL_NAMES[material_index]],
                                                isEdited=True).get_context_item_data())
    subworkflow.set_unit(scf_unit)

patch_workflow_qe_input(workflow, ADDITIONAL_PARAMETERS, unit_names=[SCF_UNIT])
visualize_workflow(workflow)

## 7. Save the materials and run the job

A job that already ran on the same three materials with the same settings is reused; otherwise a new one is created
and submitted.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in job_materials]

material_ids = [material["_id"] for material in saved_materials]
candidates = client.jobs.list({
    "_material._id": material_ids[0], "owner._id": ACCOUNT_ID, "workflow.name": workflow.name,
    "status": {"$in": ["submitted", "queued", "active", "finished"]},
})
job = next((candidate for candidate in candidates
            if [material["_id"] for material in candidate["_materials"]] == material_ids), None)
if job is None:
    job = create_job(api_client=client, materials=saved_materials, workflow=workflow, project_id=project_id,
                     owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}")
    submit_jobs(client.jobs, [job["_id"]])
    print(f"✅ Submitted job {job['_id']}")
else:
    print(f"♻️  Reusing job {job['_id']}")
await wait_for_jobs_to_finish_async(client.jobs, [job["_id"]], poll_interval=POLL_INTERVAL)
job_id = job["_id"]
job_status = client.jobs.get(job_id)["status"]
if job_status != "finished":
    raise RuntimeError(f"Job {job_id} ended with status '{job_status}'; "
                       "its page on the platform shows the failed unit.")

## 8. Retrieve the results
### 8.1. Valence band maxima and average potentials

E_VBM is the top of the highest valence band, band N_electrons / 2, along each bulk's k-path, and the gap is the
bottom of the next band minus E_VBM. V̄ is the planar average of the electrostatic potential, averaged over the whole
cell for a bulk, and over one period of the potential (two (110) planes) at the centre of each block for the
superlattice, the bulk-like region far from both interfaces.

In [ ]:
band_structures, profiles = {}, {}
for subworkflow in client.jobs.get(job_id)["workflow"]["subworkflows"]:
    units = {unit["name"]: unit for unit in subworkflow["units"]}
    name = MATERIAL_NAMES[int(next(unit["value"] for unit in subworkflow["units"]
                                   if unit["name"].startswith("Set Material Index")))]
    band_structures[name] = np.array(client.properties.get_for_job(
        job_id, property_name="band_structure", unit_id=units[BANDS_UNIT]["flowchartId"])[0]["yDataSeries"])
    profiles[name] = client.properties.get_for_job(
        job_id, property_name="average_potential_profile", unit_id=units[AVERAGE_UNIT]["flowchartId"])[0]


superlattice_in_cartesian = superlattice.clone()
superlattice_in_cartesian.to_cartesian()
heights = np.array(superlattice_in_cartesian.basis.coordinates.values)[:, 2]
elements = np.array(superlattice.basis.elements.values)
period = superlattice.lattice.c
positions = np.array(profiles[SUPERLATTICE_NAME]["xDataArray"])
planar_average = np.array(profiles[SUPERLATTICE_NAME]["yDataSeries"][0])
block_windows, superlattice_potential, bulk_potential, valence_band_maximum, bulk_gap = {}, {}, {}, {}, {}
for bulk_name in (SUBSTRATE_BULK_NAME, FILM_BULK_NAME):
    bulk = materials_by_name[bulk_name]
    cation = BLOCK_CATION[bulk_name]
    number_of_valence_bands = valence_electrons[bulk_name] // 2
    # centre of the block's planes, also when the block wraps around the cell boundary
    center = np.angle(np.exp(2j * np.pi * heights[elements == cation] / period).mean()) * period / (2 * np.pi) % period
    window = bulk.lattice.c
    block_windows[bulk_name] = (center, get_plane_spacings(bulk).mean())
    distances = (positions - center + period / 2) % period - period / 2
    superlattice_potential[bulk_name] = planar_average[np.abs(distances) <= window / 2].mean()
    bulk_potential[bulk_name] = np.mean(profiles[bulk_name]["yDataSeries"][0])
    valence_band_maximum[bulk_name] = band_structures[bulk_name][number_of_valence_bands - 1].max()
    bulk_gap[bulk_name] = band_structures[bulk_name][number_of_valence_bands].min() - valence_band_maximum[bulk_name]
    print(f"{bulk_name}: E_VBM = {valence_band_maximum[bulk_name]:.4f} eV, V̄ = {bulk_potential[bulk_name]:.4f} eV, "
          f"E_VBM - V̄ = {valence_band_maximum[bulk_name] - bulk_potential[bulk_name]:.4f} eV, "
          f"gap {bulk_gap[bulk_name]:+.3f} eV")
    print(f"  in the superlattice: V̄ = {superlattice_potential[bulk_name]:.4f} eV, averaged over "
          f"{window:.4f} Å centred at z = {center:.4f} Å")

### 8.2. Valence band offset

In [ ]:
bulk_term = ((valence_band_maximum[SUBSTRATE_BULK_NAME] - bulk_potential[SUBSTRATE_BULK_NAME])
             - (valence_band_maximum[FILM_BULK_NAME] - bulk_potential[FILM_BULK_NAME]))
potential_lineup = superlattice_potential[SUBSTRATE_BULK_NAME] - superlattice_potential[FILM_BULK_NAME]
valence_band_offset = bulk_term + potential_lineup
print(f"(E_VBM - V̄)_GaAs - (E_VBM - V̄)_AlAs = {bulk_term:+.4f} eV")
print(f"(V̄_GaAs - V̄_AlAs)_superlattice      = {potential_lineup:+.4f} eV")
print(f"ΔE_v = E_VBM(GaAs) - E_VBM(AlAs)      = {valence_band_offset:+.4f} eV")

### 8.3. Potential across the superlattice

One period from the AlAs→GaAs interface, GaAs on the left. Each bulk's own potential (dotted) is overlaid on its block,
shifted to the block's average; where they coincide the block is bulk-like, and the two averages (dashed horizontal
lines) are shifted with respect to each other by the lineup.

In [ ]:
import matplotlib.pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

substrate_center, plane_spacing = block_windows[SUBSTRATE_BULK_NAME]
# one cation per (110) plane in the primitive cell, so cations count planes
substrate_planes = np.sum(elements == BLOCK_CATION[SUBSTRATE_BULK_NAME])
plot_origin = substrate_center - substrate_planes * plane_spacing / 2  # the middle of the AlAs->GaAs interface
order = np.argsort((positions - plot_origin) % period)

figure = plt.figure(figsize=(10, 5))
plt.plot(((positions - plot_origin) % period)[order], planar_average[order], color="black", label="superlattice")
for bulk_name, color in ((SUBSTRATE_BULK_NAME, "tab:blue"), (FILM_BULK_NAME, "tab:red")):
    bulk = materials_by_name[bulk_name]
    cation = BLOCK_CATION[bulk_name]
    center, spacing = block_windows[bulk_name]
    half_width = np.sum(elements == cation) * spacing / 2
    offsets = np.linspace(-half_width, half_width, 400)  # from the block's centre
    plane_offset = (heights[elements == cation][0] - center + period / 2) % period - period / 2
    bulk_profile = profiles[bulk_name]
    bulk_in_cartesian = bulk.clone()
    bulk_in_cartesian.to_cartesian()
    bulk_plane = np.array(bulk_in_cartesian.basis.coordinates.values)[np.array(bulk.basis.elements.values) == cation][0][2]
    bulk_values = np.interp(offsets - plane_offset + bulk_plane,
                            bulk_profile["xDataArray"], bulk_profile["yDataSeries"][0], period=bulk.lattice.c)
    region = offsets + (center - plot_origin) % period
    shift = superlattice_potential[bulk_name] - bulk_potential[bulk_name]
    plt.plot(region, bulk_values + shift, color=color, linestyle=":", linewidth=2, label=f"bulk {BLOCK_LABEL[bulk_name]}, shifted")
    plt.hlines(superlattice_potential[bulk_name], region[0], region[-1], color=color, linestyle="--",
               label=f"V̄_{BLOCK_LABEL[bulk_name]} = {superlattice_potential[bulk_name]:.3f} eV")
    plt.text((center - plot_origin) % period, 0.02, BLOCK_LABEL[bulk_name], transform=plt.gca().get_xaxis_transform(), ha="center")
plt.xlim(0, period)
plt.xlabel("position along (110) (Å)")
plt.ylabel("V̄(z), electrostatic (eV)")
plt.title(f"{SUPERLATTICE_NAME}: ΔV̄ = {potential_lineup:.3f} eV, ΔE_v = {valence_band_offset:.3f} eV")
plt.legend(loc="lower right", bbox_to_anchor=(1, 0.07), fontsize=8)
display_matplotlib_figure(figure)

## 9. Compare with Hinuma et al. (2014)

Settings that differ from the manuscript: GBRV ultrasoft PBE pseudopotentials with the Ga 3d shell in the valence at
40/200 Ry, against PAW with Ga 3d frozen in the core; the lattice constants of the platform's GaAs and AlAs structures
(the paper's PBE values are not given); and the cell as built, against the relaxed one. The paper adds a spin–orbit
correction to the valence band maxima, computed with HSE; this calculation has none.

In [ ]:
HINUMA_OFFSET = 0.45  # eV, Table II, GaAs/AlAs, PBE
HINUMA_GAPS = {SUBSTRATE_BULK_NAME: 0.48, FILM_BULK_NAME: 1.32}  # eV, Table I, PBE

print(f"ΔE_v = {valence_band_offset:.3f} eV   paper {HINUMA_OFFSET:.2f} eV   "
      f"deviation {100 * (valence_band_offset / HINUMA_OFFSET - 1):+.1f} %")
print(f"GaAs valence band top {'above' if valence_band_offset > 0 else 'below'} AlAs's   paper above")
for bulk_name, paper_gap in HINUMA_GAPS.items():
    print(f"{BLOCK_LABEL[bulk_name]} gap = {bulk_gap[bulk_name]:.3f} eV   paper {paper_gap:.2f} eV   "
          f"deviation {100 * (bulk_gap[bulk_name] / paper_gap - 1):+.1f} %")
print(f"(E_VBM - V̄)_GaAs - (E_VBM - V̄)_AlAs = {bulk_term:+.2f} eV")
print(f"(V̄_GaAs - V̄_AlAs)_superlattice = {potential_lineup:+.2f} eV")

## References

[1] Y. Hinuma, A. Grüneis, G. Kresse and F. Oba, "Band alignment of semiconductors from density-functional theory and
many-body perturbation theory", Phys. Rev. B 90, 155405 (2014).
[DOI: 10.1103/PhysRevB.90.155405](https://doi.org/10.1103/PhysRevB.90.155405)

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT
calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053